# Table A

Table A represents the cleaned, point-level time series of bin fill readings derived from an ultrasonic sensor dataset.

## Data Source
The dataset used in this project is the ultrasonic waste bin sensor dataset published on Zenodo:
https://zenodo.org/records/14988663

The dataset contains fill-level measurements captured by ultrasonic sensors installed in waste bins. Sensor timestamps are generated automatically, while collection timestamps are manually entered by service providers through the management system.

For this project, we use the corrected fill files provided by the dataset authors, which include preprocessing steps to address data quality issues present in the raw sensor readings.

## Table A Schema

Table A contains the following columns:

- ContainerID
- Timestamp
- Fill_percentage
- CIDX
- REC
- Month
- Days_since_last_REC 
- Is_weekend 
- Next_day_fill_percentage

## Field Definitions
| Field | Description |
|------|-------------|
| ContainerID | Unique identifier for each waste bin.|
| Timestamp | Date and time at which the fill-level reading was recorded by the sensor system. |
| Fill_percentage | Percentage indicating how full the bin is at a given timestamp, based on the Mean fill-level value provided in the dataset.
| REC (Collection Reset Flag) | Indicates that a bin has just been emptied. A value of 1 marks the first fill reading immediately after a collection event; 0 otherwise. |
| CIDX (Cycle Index) | Identifies a single filling cycle between two collection events. All readings with the same CIDX belong to the same continuous filling period. |
| Month | Calendar month extracted from the timestamp, used to capture seasonal or monthly patterns in disposal behaviour. |
| Days_since_last_REC | The number of days elapsed since the most recent collection event for the same bin. This value resets to zero after each collection and increases over time within a fill cycle. |
| Is_weekend | Binary indicator showing whether the timestamp falls on a weekend (Saturday or Sunday), used to capture differences in disposal behaviour between weekdays and weekends. |
|Next_day_fill_percentage | The fill level of the bin recorded on the following day.|


In [16]:
# Imports & Setup
from datasets import load_dataset
from huggingface_hub import list_repo_files
import pandas as pd
import os
import logging
from datasets.utils.logging import disable_progress_bar

# to suppress Hugging Face info messages due to missing yaml metadata
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
disable_progress_bar()

In [ ]:
# Dataset Source
repo_id = "SA61team5/ultrasonic-waste-bin-sensor-raw"

all_files = list_repo_files(repo_id, repo_type="dataset")

# Select corrected fill csv files only
fill_files = sorted(
    f for f in all_files 
    if "_fill_Corrected_with_metrics" in f and f.endswith(".csv")
)

# Limit to 30 bins for this prototype
fill_files = fill_files[:30]

# Keep the container IDs of the selected files to be used in Table B
os.makedirs("Cleaned-data", exist_ok=True)
selected_container_ids = [os.path.basename(f).split("_")[1] for f in fill_files]
pd.Series(selected_container_ids, name="ContainerID").to_csv("Cleaned-data/selected_container_ids.csv", index=False)


## Per-bin Data Loading and Cleaning

For each selected bin file, we:
- Load the corrected fill data
- Extract the container identifier from the filename
- Standardise timestamps
- Rename and retain only relevant columns
- Calculate the number of days since the most recent collection event (days_since_last_REC)

In [33]:
all_fill_dfs = []

for file_path in fill_files:
    dataset = load_dataset(repo_id, data_files=file_path)
    df = pd.DataFrame(dataset["train"][:])

    # Extract container ID from filename
    container_id = os.path.basename(file_path).split("_")[1]
    df["ContainerID"] = container_id

    # Parse timestamp and derive month and is_weekend
    df["Timestamp"] = pd.to_datetime(df["Date"])
    df["Month"] = df["Timestamp"].dt.month
    df["Is_weekend"] = df["Timestamp"].dt.weekday >= 5
    df["Is_weekend"] = df["Is_weekend"].astype(int)

    df = df.rename(columns={"Mean": "Fill_percentage"})
    df = df.dropna(subset=["Fill_percentage"])

    df = df.dropna(subset=["Cidx"])

    df = df.drop(columns=["Max","Min","Fill","Date"])

    cols = ["ContainerID", "Timestamp", "Fill_percentage", "Cidx", "Rec", "Month", "Is_weekend"]

    df = df[cols]

    all_fill_dfs.append(df)

final_fill_df = pd.concat(all_fill_dfs, ignore_index=False)

# Sort by bin ID and timestamp
final_fill_df = final_fill_df.sort_values(by=["ContainerID", "Timestamp"])

# Consolidate multiple same-day readings into a single daily record per bin by keeping the maximum fill level 
# and marking the day as a collection day if any collection occurred
final_fill_df["date"] = final_fill_df["Timestamp"].dt.date

daily_df = (
    final_fill_df
    .groupby(["ContainerID", "date"], as_index=False)
    .agg({
        "Timestamp": "max",               
        "Fill_percentage": "max",           
        "Rec": "max",                       
        "Cidx": "max",                     
        "Month": "first",
        "Is_weekend": "first"
    })
)

final_fill_df = daily_df.drop(columns=["date"])

# Mark timestamps where a collection happened
final_fill_df["last_rec_timestamp"] = final_fill_df["Timestamp"].where(
    final_fill_df["Rec"] == 1
)

# Forward-fill within each bin (record when the last collection happened in every row)
final_fill_df["last_rec_timestamp"] = (final_fill_df
    .groupby("ContainerID")["last_rec_timestamp"]
    .ffill()
)

# Compute days since last collection
final_fill_df = final_fill_df.sort_values(["ContainerID", "Timestamp"]).reset_index(drop=True)
final_fill_df["Days_since_last_REC"] = ((final_fill_df["Timestamp"] - final_fill_df["last_rec_timestamp"])
    .dt.total_seconds() / (60 * 60 * 24)
)

# Cleaning up days_since_last_REC
final_fill_df["Days_since_last_REC"] = (final_fill_df["Days_since_last_REC"]
    .fillna(0)
    .clip(lower=0)
)

final_fill_df["Days_since_last_REC"] = final_fill_df["Days_since_last_REC"].astype(int)

final_fill_df = final_fill_df.drop(columns=["last_rec_timestamp"])

# Remove rows before the first recorded collection per bin
final_fill_df = final_fill_df[final_fill_df["Rec"].groupby(final_fill_df["ContainerID"]).cumsum() > 0]

# Added next-day fill percentage
final_fill_df["Next_day_fill_percentage"] = (final_fill_df
    .groupby("ContainerID")["Fill_percentage"]
    .shift(-1)
)

final_fill_df = final_fill_df.dropna(subset=["Next_day_fill_percentage"])

In [34]:
print("Shape:", final_fill_df.shape)

from IPython.display import display

display(final_fill_df.head())
display(final_fill_df.tail())

Shape: (27601, 9)


,ContainerID,Timestamp,Fill_percentage,Rec,Cidx,Month,Is_weekend,Days_since_last_REC,Next_day_fill_percentage
0,1000,2021-01-16 13:28:00,52.5,1,0.0,1,1,0,52.5
1,1000,2021-01-17 12:34:00,52.5,0,0.0,1,1,0,52.5
2,1000,2021-01-18 12:41:00,52.5,0,0.0,1,0,1,52.5
3,1000,2021-01-19 12:48:00,52.5,0,0.0,1,0,2,52.5
4,1000,2021-01-20 18:27:00,52.5,0,0.0,1,0,4,52.5


,ContainerID,Timestamp,Fill_percentage,Rec,Cidx,Month,Is_weekend,Days_since_last_REC,Next_day_fill_percentage
27625,10133,2023-08-21 14:27:00,37.0,0,29.0,8,0,4,54.0
27626,10133,2023-08-22 17:03:00,54.0,0,29.0,8,0,5,69.5
27627,10133,2023-08-23 16:28:00,69.5,0,29.0,8,0,6,100.0
27628,10133,2023-08-24 15:15:00,100.0,0,29.0,8,0,7,3.0
27629,10133,2023-08-25 13:57:00,3.0,1,30.0,8,0,0,77.0


In [35]:
# save as csv file
final_fill_df.to_csv("../Cleaned-data/tableA.csv", index=True, index_label="Index")